<a href="https://colab.research.google.com/github/LizethArista/Manejo_de_datos/blob/main/4_SOLID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a SOLID en Programación

Los principios **SOLID** son cinco reglas que ayudan a escribir código limpio, mantenible y extensible

En la práctica actuarial, necesitamos programas confiables para calcular y simular probabilidades  

Conceptos de probabilidad que usaremos:

- **Experimento aleatorio:** proceso cuyo resultado no puede predecirse con certeza (ej. lanzar un dado)

- **Evento:** conjunto de resultados posibles (ej. sacar un número par)

- **Probabilidad teórica:** valor calculado matemáticamente (ej. P(par) = 3/6 = 0.5)

- **Probabilidad empírica:** valor estimado mediante simulación

- **Distribución:** Función que asigna probabilidades a resultados (por ejemplo Normal, Binomial, Poisson, Exponencial

Objetivo: aplicar SOLID en ejemplos de probabilidad para que el código sea fácil de probar y extender.


> SOLID nos permite separar el *modelo matemático* del *reporte* y de la *fuente de datos*.

# Principio S: Responsabilidad Única (SRP)

> *"Una clase debe tener una sola razón para cambiar."*


Ejemplo: separar cálculo de probabilidad, simulación y reporte.

En estadística, **no confundimos** el *Generador del Proceso* (la regla matemática) con el *Reporte* (la tabla o gráfica). Mezclarlos hace que cambiar el formato de salida pueda romper la fórmula matemática.
Estimaremos la probabilidad de obtener un número par al lanzar un dado.

-  **Mala práctica:** Una clase que simula, calcula media y además imprime en pantalla.
-  **Buena práctica:**
  - `MotorBernoulli` → solo hace matemáticas
  - `ReporteConsola` → solo formatea y muestra

Si se quiere exportar a Excel, **solo** cambia el `Reporte`, nunca la matemática

In [ ]:
import random

# MALA PRÁCTICA (Violación de SRP)
# Esta clase hace matemáticas Y también se encarga de la salida visual.
class SimuladorYReporte:
    def ejecutar_y_mostrar(self, p, n):
        exitos = sum(1 for _ in range(n) if random.random() < p)
        prob = exitos / n
        print(f"--- REPORTE ACTUARIAL ---\nProbabilidad: {prob:.4f}")

# BUENA PRÁCTICA (Aplicando SRP)

# 1. Clase exclusiva para la matemática (Motor de Simulación)
class MotorBernoulli:
    """Calcula la probabilidad empírica de éxito en un ensayo de Bernoulli."""
    def __init__(self, p: float):
        self.p = p  # Probabilidad teórica de éxito

    def simular(self, n_simulaciones: int) -> float:
        """Genera n experimentos y retorna la frecuencia relativa."""
        exitos = sum(1 for _ in range(n_simulaciones) if random.random() < self.p)
        return exitos / n_simulaciones

# 2. Clase exclusiva para la presentación (Reporte)
class ReporteConsola:
    """Formatea y muestra resultados """
    def mostrar(self, etiqueta: str, valor: float):
        print(f"[{etiqueta}] Resultado estimado: {valor:.4f}")

# --- USO ---
motor = MotorBernoulli(p=0.3)
resultado = motor.simular(n_simulaciones=10000)
ReporteConsola().mostrar("Bernoulli p=0.3", resultado)

[Bernoulli p=0.3] Resultado estimado: 0.2906


# Principio O: Abierto/Cerrado (OCP)

> *"Abierto a extensión, cerrado a modificación."*

Ejemplo: agregar nuevas distribuciones de probabilidad sin modificar código existente.


Existen **familias de distribuciones** (Discretas, Continuas)

- Modelar **frecuencia** de siniestros → Binomial / Poisson

- Modelar **severidad** → Exponencial / LogNormal

El motor de análisis **no debe romperse** al cambiar la familia, debemos tener en cuenta que cada distribución tiene su propia forma de generar resultados


Usamos **Polimorfismo** con una clase abstracta `Distribucion`
- El motor (`ejecutar_experimento`) **nunca cambia**.
- Para agregar una nueva distribución (ej. `Exponencial`), **solo creamos una nueva clase**


In [ ]:
from abc import ABC, abstractmethod
import math

# 1. Contrato (Abstracción)
class Distribucion(ABC):
    """Clase base abstracta. Obliga a toda distribución a tener simular()"""
    @abstractmethod
    def simular(self) -> float:
        pass

# 2. Extensiones existentes
class Dado(Distribucion):
    def simular(self) -> int:
        return random.randint(1, 6)

class Bernoulli(Distribucion):
    def __init__(self, p: float):
        self.p = p
    def simular(self) -> int:
        return 1 if random.random() < self.p else 0

class Binomial(Distribucion):
    def __init__(self, n: int, p: float):
        self.n = n
        self.p = p
    def simular(self) -> int:
        return sum(1 for _ in range(self.n) if random.random() < self.p)

#  NUEVA DISTRIBUCIÓN: Exponencial
# para modelar "tiempo entre siniestros" o "vida útil".
class Exponencial(Distribucion):
    """
    Simula una variable aleatoria continua Exponencial.
    Método de la Transformada Inversa: X = -ln(U) / lambda
    """
    def __init__(self, lambda_param: float):
        self.lambda_param = lambda_param  # Tasa de ocurrencia (1/Esperanza)

    def simular(self) -> float:
        u = random.random()  # U ~ Uniforme(0,1)
        while u == 0.0:      # Evitamos log(0)
            u = random.random()
        return -math.log(u) / self.lambda_param

# --- MOTOR DE EXPERIMENTOS (Cerrado a modificación) ---
def ejecutar_experimento(dist: Distribucion, n: int = 1000) -> float:
    """Calcula el valor esperado empírico (media muestral)."""
    resultados = [dist.simular() for _ in range(n)]
    return sum(resultados) / n

# --- USO ---
print("Media Dado:", ejecutar_experimento(Dado(), 5000))                    # Esperado ~3.5
print("Media Bernoulli:", ejecutar_experimento(Bernoulli(0.3), 5000))       # Esperado ~0.3
print("Media Binomial(10,0.5):", ejecutar_experimento(Binomial(10, 0.5), 5000))  # Esperado ~5
print("Media Exponencial (lambda=0.5):", ejecutar_experimento(Exponencial(0.5), 5000))  # Esperado ~2.0

Media Dado: 3.507
Media Bernoulli: 0.2996
Media Binomial(10,0.5): 5.0154
Media Exponencial (lambda=0.5): 1.9631168555470517


# Principio L: Sustitución de Liskov (LSP)

> *"Las subclases deben poder sustituir a la clase base sin alterar el comportamiento."*

Si una función espera calcular la **Esperanza Matemática** (media) de un proceso, asume que el proceso retornará **números reales válidos**. Si una subclase retorna `None`, un string, o lanza excepción, rompe la *Ley de los Grandes Números* en nuestro código.

- Lo correto es que todas las distribuciones devuelven **números** → el motor estadístico funciona.
- Si una distribución que devuelve strings → `sum()` colapsa

El LSP garantiza que nuestro motor es **matemáticamente ciego a los detalles internos**, pero **confiable en el tipo de dato**


In [ ]:
#  VIOLACIÓN DE LSP (Ejemplo teórico de lo que NO se debe hacer)
class DistribucionRota(Distribucion):
    """
    Viola el LSP: simular() no devuelve un número, sino un string.
    Si la pasamos a ejecutar_experimento(), el programa colapsará.
    """
    def simular(self):
        return "No sé simular esto"

#  CUMPLIMIENTO DE LSP
# Todas nuestras clases devuelven números, podemos usarlas en funciones genéricas

def calcular_varianza_empirica(dist: Distribucion, n: int = 1000) -> float:
    """
    Función genérica de estadística.
    Gracias al LSP, sabemos que dist.simular() SIEMPRE devolverá un número.
    """
    muestras = [dist.simular() for _ in range(n)]
    media = sum(muestras) / n
    varianza = sum((x - media) ** 2 for x in muestras) / (n - 1)
    return varianza

# El LSP nos da tranquilidad matemática
print("Varianza Dado:", calcular_varianza_empirica(Dado()))
print("Varianza Bernoulli:", calcular_varianza_empirica(Bernoulli(0.3)))
print("Varianza Exponencial:", calcular_varianza_empirica(Exponencial(0.5)))

# Si intentáramos pasar DistribucionRota(), el sum() fallaría:
# calcular_varianza_empirica(DistribucionRota())  # TypeError

Varianza Dado: 2.9197357357357356
Varianza Bernoulli: 0.19821421421421423
Varianza Exponencial: 4.142375679738723


# Principio I: Segregación de Interfaces (ISP)

> *"Varias interfaces específicas son mejores que una interfaz general."*
Ejemplo: separar interfaces de simulación y cálculo de probabilidad teórica

Concepto de probabilidad: diferencia entre probabilidad teórica (calculada) y probabilidad empírica (simulada).

No todas las distribuciones tienen fórmula cerrada para su **Probabilidad Teórica** $P(X=k)$:
-  Dado, Bernoulli, Binomial → tienen fórmula exacta
-  Modelos de Montecarlo complejos, ML → **solo se pueden simular**.

Si creamos una clase abstracta que exija `simular()` **Y** `prob_teorica()`, obligaríamos a los modelos puramente empíricos a implementar métodos imposibles (`NotImplementedError`).

**Solución:** Dividimos las interfaces:
- `Simulable` → para Monte Carlo
- `ProbabilidadExacta` → para modelos con fórmula cerrada

In [ ]:
from typing import Protocol

# 1. Interfaces pequeñas y específicas (Protocolos en Python)
class Simulable(Protocol):
    """Contrato para modelos que pueden generar datos empíricos (Monte Carlo)"""
    def simular(self) -> float: ...

class ProbabilidadExacta(Protocol):
    """Contrato para modelos que tienen fórmula cerrada (Teórica)"""
    def prob_teorica(self, evento) -> float: ...

# 2. Clase que cumple AMBAS (El Dado tiene fórmula y se puede simular)
class DadoAvanzado:
    def simular(self) -> int:
        return random.randint(1, 6)

    def prob_teorica(self, evento: int) -> float:
        return 1/6 if 1 <= evento <= 6 else 0.0

# 3. Clase que SOLO cumple Simulable (Montecarlo complejo sin fórmula cerrada)
class SimuladorMontecarloComplejo:
    def __init__(self, semilla: int):
        random.seed(semilla)

    def simular(self) -> float:
        # Simulación compleja sin fórmula cerrada conocida
        return sum(random.gauss(0, 1) for _ in range(100))

    # NO implementamos prob_teorica() porque es matemáticamente imposible.
    # El ISP nos salva de tener que escribir "raise NotImplementedError"

# --- USO SEGURO ---
def correr_simulacion(modelo: Simulable, n: int):
    return [modelo.simular() for _ in range(n)]

def verificar_teorico(modelo: ProbabilidadExacta, valor):
    return modelo.prob_teorica(valor)

dado = DadoAvanzado()
monte_carlo = SimuladorMontecarloComplejo(semilla=42)

# Ambos pueden simularse
print("Simulando Dado:", correr_simulacion(dado, 3))
print("Simulando Monte Carlo:", correr_simulacion(monte_carlo, 3))

# Pero solo el dado puede usarse en funciones teóricas
print("Teórica Dado (sacar 3):", verificar_teorico(dado, 3))
# verificar_teorico(monte_carlo, 3)  # El linter nos avisaría que es un error

Simulando Dado: [6, 1, 1]
Simulando Monte Carlo: [0.3467250496186626, -7.530638192202717, -2.7510246477768376]
Teórica Dado (sacar 3): 0.16666666666666666


# Principio D: Inversión de Dependencias (DIP)

> *"Depender de abstracciones, no de implementaciones concretas."*

Ejemplo: un servicio que calcula probabilidades usando un repositorio abstracto

Concepto de probabilidad: podemos obtener datos de distintas fuentes (simulación, encuestas, bases de datos)

Para calcular el **Value at Risk (VaR)** o la prima de un seguro, necesitamos datos. Estos pueden venir de:

1. **Simulación** (Generador pseudoaleatorio).

2. **Datos Históricos** (Base de datos SQL de siniestros pasados)

3. **API Externa** (Datos de mercado en tiempo real).

El motor de cálculo **no debe acoplarse** a ninguna fuente


- `MotorActuarial` (alto nivel) **NO** importa `random` ni `sqlite3`
- Exige un `RepositorioDeDatos` (abstracción)
- Si mañana cambiamos de simulación a datos reales de producción, **el motor no sufre ningún cambio**

In [ ]:
# 1. Abstracción (La interfaz que el Motor exige)
class RepositorioDeDatos(ABC):
    """Contrato para obtener muestras de datos (reales o simuladas)."""
    @abstractmethod
    def obtener_muestra(self, tamano: int) -> list:
        pass

# 2. Implementaciones de bajo nivel (Las fuentes concretas)
class RepositorioSimulado(RepositorioDeDatos):
    """Genera datos al vuelo usando una distribución."""
    def __init__(self, distribucion: Distribucion):
        self.distribucion = distribucion

    def obtener_muestra(self, tamano: int) -> list:
        return [self.distribucion.simular() for _ in range(tamano)]

class RepositorioHistoricoCSV(RepositorioDeDatos):
    """Simula leer de una base de datos (en producción leería un CSV/SQL)"""
    def __init__(self, datos_ficticios: list):
        self.datos_ficticios = datos_ficticios  # Ej. Siniestros del año pasado

    def obtener_muestra(self, tamano: int) -> list:
        return self.datos_ficticios[:tamano]

# 3. Módulo de Alto Nivel (El Motor Actuarial)
# NOTA EL DIP: No importa 'random', ni 'csv', ni ninguna distribución concreta
class MotorActuarial:
    def __init__(self, repo: RepositorioDeDatos):
        self.repo = repo  # Inyección de Dependencias

    def calcular_prima_riesgo(self, tamano_muestra: int = 1000) -> float:
        """Calcula la prima esperada basándose en los datos del repositorio."""
        datos = self.repo.obtener_muestra(tamano_muestra)
        if not datos:
            return 0.0
        # Fórmula simple: Esperanza empírica + Carga de seguridad (20%)
        esperanza = sum(datos) / len(datos)
        return esperanza * 1.20

# --- USO ---
# Escenario A: Fase de pruebas, usamos simulación
repo_sim = RepositorioSimulado(Exponencial(lambda_param=0.1))  # Severidad media 10
motor_sim = MotorActuarial(repo_sim)
print("Prima (Simulada):", motor_sim.calcular_prima_riesgo())

# Escenario B: Producción, usamos datos históricos reales
datos_reales = [15.2, 8.4, 22.1, 5.0, 11.3]  # En miles de dólares
repo_hist = RepositorioHistoricoCSV(datos_reales)
motor_hist = MotorActuarial(repo_hist)
print("Prima (Histórica):", motor_hist.calcular_prima_riesgo(tamano_muestra=5))

Prima (Simulada): 11.427087634680987
Prima (Histórica): 14.879999999999999


# **Ejemplo 2**

##  Anualidades



In [ ]:
# =========================================================
# Utilidades financieras base
# =========================================================
from abc import ABC, abstractmethod
from typing import List, Protocol
from dataclasses import dataclass


def factor_descuento(i: float, n: int) -> float:
    return 1 / (1 + i) ** n


def factor_acumulado(i: float, n: int) -> float:
    return (1 + i) ** n


@dataclass
class Flujo:
    """Un pago en un momento del tiempo."""
    tiempo: int
    monto: float

## S — Responsabilidad Única
Separar **qué se paga** (generador) de **cómo se valúa** (valuador).

In [ ]:
# =========================================================
# SRP: Generador de flujos  ≠  Valuador
# =========================================================

class GeneradorFlujos(ABC):
    @abstractmethod
    def generar(self) -> List[Flujo]:
        pass


class AnualidadVencida(GeneradorFlujos):
    def __init__(self, pago: float, n: int):
        self.pago, self.n = pago, n
    def generar(self) -> List[Flujo]:
        return [Flujo(t, self.pago) for t in range(1, self.n + 1)]


class AnualidadAnticipada(GeneradorFlujos):
    def __init__(self, pago: float, n: int):
        self.pago, self.n = pago, n
    def generar(self) -> List[Flujo]:
        return [Flujo(t, self.pago) for t in range(0, self.n)]


class ValuadorFlujos:
    """Solo descuenta flujos a valor presente."""
    def __init__(self, tasa: float):
        self.tasa = tasa
    def valor_presente(self, flujos: List[Flujo]) -> float:
        return sum(f.monto * factor_descuento(self.tasa, f.tiempo) for f in flujos)


# --- Uso ---
flujos = AnualidadAnticipada(1000, 5).generar()
vp = ValuadorFlujos(0.08).valor_presente(flujos)
print(f"VP anualidad anticipada: ${vp:,.2f}")

VP anualidad anticipada: $4,312.13


## O — Abierto/Cerrado
Agregar nuevas formas de valuar (VF, valor en un instante) sin modificar el motor.

In [ ]:
# =========================================================
# OCP: Estrategias de valuación
# =========================================================

class EstrategiaValuacion(ABC):
    @abstractmethod
    def valuar(self, flujos: List[Flujo], tasa: float) -> float:
        pass


class ValuarEnPresente(EstrategiaValuacion):
    def valuar(self, flujos: List[Flujo], tasa: float) -> float:
        return sum(f.monto * factor_descuento(tasa, f.tiempo) for f in flujos)


class ValuarEnFuturo(EstrategiaValuacion):
    def valuar(self, flujos: List[Flujo], tasa: float) -> float:
        n = max(f.tiempo for f in flujos)
        return sum(f.monto * factor_acumulado(tasa, n - f.tiempo) for f in flujos)


# --- Motor cerrado a modificación ---
def valuar(flujos: List[Flujo], tasa: float, estrategia: EstrategiaValuacion) -> float:
    return estrategia.valuar(flujos, tasa)


# --- Uso ---
flujos = AnualidadVencida(1000, 5).generar()
i = 0.08
print("VP:", f"${valuar(flujos, i, ValuarEnPresente()):,.2f}")
print("VF:", f"${valuar(flujos, i, ValuarEnFuturo()):,.2f}")

VP: $3,992.71
VF: $5,866.60


## L — Sustitución de Liskov
Toda anualidad debe cumplir la **identidad fundamental**:

$$VP \cdot (1+i)^n = VF$$

Si una subclase viola esto, no es una anualidad válida.

In [ ]:
# =========================================================
# LSP: Invariante VP · (1+i)^n = VF
# =========================================================

class Anualidad(ABC):
    @abstractmethod
    def vp(self, i: float) -> float: ...
    @abstractmethod
    def vf(self, i: float) -> float: ...
    @abstractmethod
    def n_periodos(self) -> int: ...


class AnualidadVencidaModelo(Anualidad):
    def __init__(self, R: float, n: int):
        self.R, self.n = R, n
    def vp(self, i: float) -> float:
        return self.R * (1 - factor_descuento(i, self.n)) / i
    def vf(self, i: float) -> float:
        return self.R * (factor_acumulado(i, self.n) - 1) / i
    def n_periodos(self) -> int:
        return self.n


def verificar_identidad(a: Anualidad, i: float) -> bool:
    """Verifica VP · (1+i)^n == VF"""
    return abs(a.vp(i) * factor_acumulado(i, a.n_periodos()) - a.vf(i)) < 1e-6


# --- Uso ---
a = AnualidadVencidaModelo(1000, 5)
print("Cumple identidad:", verificar_identidad(a, 0.08))

Cumple identidad: True


## I — Segregación de Interfaces
Dos interfaces pequeñas:
- `Valuable` → calcular VP / VF
- `Simulable` → estimar por Monte Carlo (cuando la tasa es aleatoria)

In [ ]:
# =========================================================
# ISP: Dos interfaces pequeñas
# =========================================================
import random

class Valuable(Protocol):
    def vp(self, i: float) -> float: ...
    def vf(self, i: float) -> float: ...

class Simulable(Protocol):
    def simular_vp(self, n_sim: int = 1000) -> float: ...


# --- Anualidad con tasa fija (Valuable) ---
class AnualidadTasaFija:
    def __init__(self, pago: float, i: float, n: int):
        self.pago, self.i, self.n = pago, i, n
    def vp(self, i: float) -> float:
        return self.pago * (1 - factor_descuento(i, self.n)) / i
    def vf(self, i: float) -> float:
        return self.vp(i) * factor_acumulado(i, self.n)


# --- Anualidad con tasa aleatoria (solo Simulable) ---
class AnualidadTasaVariable:
    def __init__(self, pago: float, media_i: float, n: int):
        self.pago, self.media_i, self.n = pago, media_i, n
    def simular_vp(self, n_sim: int = 1000) -> float:
        total = 0.0
        for _ in range(n_sim):
            vp = 0.0
            for t in range(1, self.n + 1):
                i_t = random.gauss(self.media_i, 0.01)
                vp += self.pago / (1 + i_t) ** t
            total += vp
        return total / n_sim


# --- Uso ---
exacta = AnualidadTasaFija(1000, 0.08, 5)
estocastica = AnualidadTasaVariable(1000, 0.08, 5)
print(f"VP exacto:    ${exacta.vp(0.08):,.2f}")
print(f"VP simulado:  ${estocastica.simular_vp():,.2f}")

VP exacto:    $3,992.71
VP simulado:  $3,995.79


## D — Inversión de Dependencias
El motor no conoce `round()`. Depende de una `EstrategiaRedondeo`.

In [ ]:
# =========================================================
# DIP: Estrategia de redondeo
# =========================================================

class EstrategiaRedondeo(ABC):
    @abstractmethod
    def aplicar(self, monto: float) -> float:
        pass


class SinRedondeo(EstrategiaRedondeo):
    def aplicar(self, monto: float) -> float:
        return monto

class RedondeoCentavos(EstrategiaRedondeo):
    def aplicar(self, monto: float) -> float:
        return round(monto, 2)


class MotorAnualidad:
    def __init__(self, pago: float, i: float, n: int, redondeo: EstrategiaRedondeo):
        self.pago, self.i, self.n = pago, i, n
        self.redondeo = redondeo

    def vp(self) -> float:
        raw = self.pago * (1 - factor_descuento(self.i, self.n)) / self.i
        return self.redondeo.aplicar(raw)


# --- Uso ---
pago, i, n = 1234.5678, 0.08, 5
print("Sin redondeo:", MotorAnualidad(pago, i, n, SinRedondeo()).vp())
print("A centavos:  ", MotorAnualidad(pago, i, n, RedondeoCentavos()).vp())

Sin redondeo: 4929.271246513415
A centavos:   4929.27


In [ ]:
# UN EJEMPLO MAS DE COMO FUNCIONA PROTOCOL... IMPLEMETACIÓN

# CONTRATO (no requiere herencia)
class Valuable(Protocol):
    def vp(self, i: float) -> float: ...

# Clase que CUMPLE el contrato SIN heredar
class AnualidadTasaFija:
    def __init__(self, pago: float, i: float, n: int):
        self.pago, self.i, self.n = pago, i, n

    def vp(self, i: float) -> float:
        return self.pago * (1 - (1+i)**(-self.n)) / i

# Función que espera un Valuable
def calcular(x: Valuable) -> float:
    return x.vp(0.08)

# Uso: funciona aunque NO haya herencia
obj = AnualidadTasaFija(1000, 0.08, 5)
print(f"VP = ${calcular(obj):,.2f}")

VP = $3,992.71
